# Daily Challenge: Fine-Tune an LLM with LoRA

This notebook fine-tunes **bigscience/bloomz-560m** on a sample of English quotes using Parameter-Efficient Fine-Tuning (PEFT / LoRA).

## Step 1 – Install libraries

In [ ]:
%pip install --quiet peft==0.4.0 datasets transformers accelerate

## Step 2 – Load the pre-trained model and tokenizer

In [ ]:
import os
os.makedirs("cache", exist_ok=True)

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

# Load tokenizer and set pad token so DataCollator works correctly
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # BLOOM has no pad token by default

# Load the base causal-LM
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)
print(f"Model loaded: {model_name}")
print(f"Parameters: {foundation_model.num_parameters():,}")

## Step 3 – Load and preprocess the dataset

In [ ]:
from datasets import load_dataset

# Load the dataset and sample 10% of the training split
data = load_dataset("Abirate/english_quotes", split="train")
data = data.select(range(int(len(data) * 0.10)))   # 10% sample

# Tokenise the "quote" column; truncate to 512 tokens to avoid OOM
def tokenize(samples):
    return tokenizer(samples["quote"], truncation=True, max_length=512)

data = data.map(tokenize, batched=True)

# Use only the first 5 examples for the quick training demo
train_sample = data.select(range(5))
print(f"Full 10% sample size : {len(data)}")
print(f"Training subset size : {len(train_sample)}")
train_sample.to_pandas()[["quote", "input_ids"]].head()

## Step 4 & 5 – Configure LoRA and apply it to the model

In [ ]:
from peft import LoraConfig, get_peft_model

# Identify the linear projection layers inside BLOOM attention blocks
# (query_key_value is the fused QKV projection used by BLOOM)
lora_config = LoraConfig(
    r=1,                            # low-rank dimension (kept small for speed)
    lora_alpha=1,                   # scaling factor; set equal to r as a starting point
    target_modules=["query_key_value"],  # BLOOM's fused attention projection
    lora_dropout=0.05,              # small dropout for regularisation
    bias="none",                    # do not train bias terms
    task_type="CAUSAL_LM"           # causal language modelling
)

# Wrap the foundation model with LoRA adapters
peft_model = get_peft_model(foundation_model, lora_config)
peft_model.print_trainable_parameters()

## Steps 6 & 7 – Training arguments and Trainer

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("cache", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)

training_args = TrainingArguments(
    report_to="none",              # disable W&B / MLflow logging
    output_dir=output_directory,
    auto_find_batch_size=True,     # automatically finds the largest batch that fits in memory
    learning_rate=3e-2,            # higher LR is typical for PEFT / LoRA
    num_train_epochs=1,            # 1 epoch for the demo; increase for real use
    use_cpu=True                   # remove this line (or set False) when a GPU is available
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
    # mlm=False -> causal LM objective (next-token prediction)
)

trainer.train()

## Step 8 – Save the fine-tuned LoRA adapter

In [ ]:
import time

# Use a timestamp to version the saved adapter
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# save_pretrained writes only the LoRA adapter weights (~KB), not the full model
trainer.model.save_pretrained(peft_model_path)
print(f"Adapter saved to: {peft_model_path}")
print("Files:", os.listdir(peft_model_path))

## Step 9 – Reload the saved adapter for inference

In [ ]:
from peft import PeftModel

# Reload the base model and attach the saved LoRA adapter (frozen)
base_model_reloaded = AutoModelForCausalLM.from_pretrained(model_name)
loaded_peft_model = PeftModel.from_pretrained(
    base_model_reloaded,
    peft_model_path,
    is_trainable=False    # inference only; no further training
)
loaded_peft_model.eval()
print("LoRA adapter loaded successfully.")

## Step 10 – Generate text with the fine-tuned model

In [ ]:
import torch

prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = loaded_peft_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=50,         # how many tokens to generate beyond the prompt
        do_sample=True,            # sample for varied output
        temperature=0.7,           # controls randomness (lower = more deterministic)
        top_p=0.9,                 # nucleus sampling
        pad_token_id=tokenizer.eos_token_id   # suppress pad-token warnings
    )

generated = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print("Generated text:")
for text in generated:
    print("-", text)

---
## Summary

| Step | What happened |
|---|---|
| Load model | `bigscience/bloomz-560m` loaded from Hugging Face Hub |
| Dataset | `Abirate/english_quotes` – 10% sample, tokenised on the `quote` column |
| LoRA config | `r=1`, `alpha=1`, target = `query_key_value` (BLOOM's fused QKV layer) |
| Trainable params | ~0.07% of total (LoRA only) |
| Training | 1 epoch, `lr=3e-2`, CPU, 5-example subset |
| Saved artifact | Adapter weights only (a few KB vs ~1 GB for the full model) |
| Inference | Reload with `PeftModel.from_pretrained(..., is_trainable=False)`, then `generate()` |